In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                              recall_score, confusion_matrix)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('outputs', exist_ok=True)

# Load clean data — we need all raw indicators
df_raw = pd.read_csv('data/processed/nepal_macro_clean.csv',
                      index_col=0, parse_dates=True)

# Load current best model features for comparison
df_current = pd.read_csv('data/processed/nepal_model_features.csv',
                          index_col=0, parse_dates=True)

print(f"✅ Raw data loaded: {df_raw.shape}")
print(f"   Columns: {list(df_raw.columns)}")
print(f"\n✅ Current model features: {df_current.shape}")
print(f"   Features: {[c for c in df_current.columns if c != 'distress']}")

✅ Raw data loaded: (31, 12)
   Columns: ['gdp_growth', 'inflation', 'unemployment', 'remittances_pct_gdp', 'forex_reserves_months', 'gross_investment_gdp', 'consumption_growth', 'exports_pct_gdp', 'imports_pct_gdp', 'india_gdp_growth', 'india_inflation', 'distress']

✅ Current model features: (30, 6)
   Features: ['unemployment_roll3_mean', 'gdp_growth_roll3_std', 'consumption_growth_roll3_mean', 'forex_rapid_decline', 'remittances_pct_gdp_lag1']


In [2]:
# ================================================
# 3 NEW FEATURES TO REPLACE UNEMPLOYMENT
# ================================================

df = df_raw.copy()

# ---- Feature 1: External Pressure Index ----
# Combines forex decline + import surge
# Nepal's 2021-22 crisis = reserves falling
# while imports were surging simultaneously
# Formula: forex monthly drop + import growth rate
df['external_pressure'] = (
    -df['forex_reserves_months'].diff(1) +
    df['imports_pct_gdp'].pct_change(1) * 10
)
# Negative forex change = pressure rising
# High import growth = pressure rising
# Both together = serious external stress

# ---- Feature 2: Inflation-Growth Divergence ----
# When prices rise but output doesn't = stagflation
# Nepal 2015-16: inflation high, growth near zero
# Nepal 2022-23: high inflation + slow recovery
# Formula: inflation minus GDP growth (lagged 1yr)
df['inflation_growth_gap'] = (
    df['inflation'] - df['gdp_growth'].shift(1)
)
# Positive = inflation outpacing growth = stress
# Negative = growth outpacing inflation = healthy

# ---- Feature 3: Remittance-Forex Stress ----
# Nepal-specific: volatile remittances + thin forex
# = double vulnerability unique to Nepal's structure
# Formula: remittance volatility / forex adequacy
df['remittance_forex_stress'] = (
    df['remittances_pct_gdp'].rolling(3).std() /
    df['forex_reserves_months'].rolling(3).mean()
)
# High = remittances unstable AND forex reserves low
# Low = remittances stable AND forex reserves adequate

# ---- Preview all 3 features ----
new_features = ['external_pressure',
                'inflation_growth_gap',
                'remittance_forex_stress']

print("✅ 3 new features created!")
print("\nFeature statistics:")
for feat in new_features:
    cv = df[feat].std() / abs(df[feat].mean())
    corr = abs(df[feat].corr(df['distress']))
    print(f"\n  {feat}")
    print(f"    Range:       {df[feat].min():.2f} to {df[feat].max():.2f}")
    print(f"    CV:          {cv:.3f}  (higher = more variation = better)")
    print(f"    Corr w distress: {corr:.3f}")

# Compare with unemployment
unemp_cv   = df['unemployment'].std() / df['unemployment'].mean()
unemp_corr = abs(df['unemployment'].corr(df['distress']))
print(f"\n  unemployment (OLD for comparison)")
print(f"    CV:              {unemp_cv:.3f}")
print(f"    Corr w distress: {unemp_corr:.3f}")

✅ 3 new features created!

Feature statistics:

  external_pressure
    Range:       -8.16 to 6.88
    CV:          7.008  (higher = more variation = better)
    Corr w distress: 0.143

  inflation_growth_gap
    Range:       -4.57 to 6.50
    CV:          3.267  (higher = more variation = better)
    Corr w distress: 0.132

  remittance_forex_stress
    Range:       0.00 to 0.73
    CV:          0.910  (higher = more variation = better)
    Corr w distress: 0.015

  unemployment (OLD for comparison)
    CV:              0.050
    Corr w distress: 0.498


In [3]:
# ================================================
# BUILD NEW FEATURE DATASETS AND COMPARE
# ================================================

# Align index with current model dataset
df_aligned = df.loc[df_current.index]

# Keep the 4 existing good features
feat_keep = [
    'gdp_growth_roll3_std',
    'consumption_growth_roll3_mean',
    'forex_rapid_decline',
    'remittances_pct_gdp_lag1'
]

# Load these from current model dataset
df_base = df_current[feat_keep + ['distress']].copy()

# Add new features from raw data
df_base['external_pressure']      = df_aligned['external_pressure']
df_base['inflation_growth_gap']    = df_aligned['inflation_growth_gap']
df_base['remittance_forex_stress'] = df_aligned['remittance_forex_stress']

# Drop any NaN
df_base = df_base.dropna()

print(f"Dataset shape: {df_base.shape}")
print(f"Missing values: {df_base.isnull().sum().sum()}")
print(f"Distress rate: {df_base['distress'].mean():.1%}")

# ---- LOOCV evaluation function ----
def run_loocv(X, y, model, label):
    loo = LeaveOneOut()
    preds, actuals = [], []
    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_train)
        X_te_s = scaler.transform(X_test)
        model.fit(X_tr_s, y_train)
        preds.append(model.predict(X_te_s)[0])
        actuals.append(y_test.values[0])
    f1  = f1_score(actuals, preds, zero_division=0)
    bal = balanced_accuracy_score(actuals, preds)
    rec = recall_score(actuals, preds, zero_division=0)
    # Count false alarms
    fa = sum(1 for a,p in zip(actuals,preds) if a==0 and p==1)
    print(f"\n{label}")
    print(f"  F1={f1:.3f}  BalAcc={bal:.3f}  Recall={rec:.3f}  FalseAlarms={fa}")
    return {'label':label,'f1':f1,'bal':bal,'rec':rec,'fa':fa}

xgb = XGBClassifier(n_estimators=100, max_depth=3,
                    learning_rate=0.1, scale_pos_weight=3,
                    random_state=42, eval_metric='logloss',
                    verbosity=0)

print("\n📊 TESTING ALL COMBINATIONS:")

# Option 1 — Original (with unemployment)
X1 = df_current.drop(columns=['distress']).loc[df_base.index]
y1 = df_current['distress'].loc[df_base.index]
r1 = run_loocv(X1, y1, xgb, "Option 1: Original (with unemployment)")

# Option 2 — Replace unemployment with all 3 new features
X2 = df_base[feat_keep + ['external_pressure',
                           'inflation_growth_gap',
                           'remittance_forex_stress']]
y2 = df_base['distress']
r2 = run_loocv(X2, y2, xgb, "Option 2: Replace unemployment with 3 new features")

# Option 3 — Keep unemployment + add best new feature
X3 = df_current[feat_keep + ['unemployment_roll3_mean']].loc[df_base.index].copy()
X3['external_pressure'] = df_base['external_pressure']
y3 = df_current['distress'].loc[df_base.index]
r3 = run_loocv(X3, y3, xgb, "Option 3: Keep unemployment + add external_pressure")

# Option 4 — Replace unemployment with only external_pressure
X4 = df_base[feat_keep + ['external_pressure']]
y4 = df_base['distress']
r4 = run_loocv(X4, y4, xgb, "Option 4: Replace unemployment with external_pressure only")

Dataset shape: (29, 8)
Missing values: 0
Distress rate: 24.1%

📊 TESTING ALL COMBINATIONS:

Option 1: Original (with unemployment)
  F1=0.714  BalAcc=0.812  Recall=0.714  FalseAlarms=2

Option 2: Replace unemployment with 3 new features
  F1=0.400  BalAcc=0.601  Recall=0.429  FalseAlarms=5

Option 3: Keep unemployment + add external_pressure
  F1=0.571  BalAcc=0.718  Recall=0.571  FalseAlarms=3

Option 4: Replace unemployment with external_pressure only
  F1=0.429  BalAcc=0.623  Recall=0.429  FalseAlarms=4


In [4]:
print("="*55)
print("  FINAL MODEL DECISION — LOCKED")
print("="*55)
print("  Model:    XGBoost")
print("  Features: 5 original features")
print("  F1:       0.714")
print("  BalAcc:   0.814")
print("  Recall:   0.714")
print("  FalseAlarms: 2")
print()
print("  Features:")
for f in df_current.drop(columns=['distress']).columns:
    print(f"    → {f}")
print()
print("  Decision: Keep original unemployment feature")
print("  Reason:   COVID spike provides strongest signal")
print("  Note:     Limitation documented in paper Section 6")
print("="*55)

  FINAL MODEL DECISION — LOCKED
  Model:    XGBoost
  Features: 5 original features
  F1:       0.714
  BalAcc:   0.814
  Recall:   0.714
  FalseAlarms: 2

  Features:
    → unemployment_roll3_mean
    → gdp_growth_roll3_std
    → consumption_growth_roll3_mean
    → forex_rapid_decline
    → remittances_pct_gdp_lag1

  Decision: Keep original unemployment feature
  Reason:   COVID spike provides strongest signal
  Note:     Limitation documented in paper Section 6


In [5]:
# ================================================
# TEST ADDING MORE FEATURES SYSTEMATICALLY
# ================================================

from itertools import combinations

# Pool of candidate features to add
# (from our 27-feature shortlist, excluding
#  what's already in the model)
df_27 = pd.read_csv('data/processed/nepal_final_features.csv',
                     index_col=0, parse_dates=True)

# Current 5 features
current_5 = [
    'unemployment_roll3_mean',
    'gdp_growth_roll3_std',
    'consumption_growth_roll3_mean',
    'forex_rapid_decline',
    'remittances_pct_gdp_lag1'
]

# Candidate features to try adding
# Pick from our verified 27 — exclude current 5
candidates = [c for c in df_27.columns
              if c != 'distress'
              and c not in current_5]

# Check correlations of candidates with distress
candidate_corrs = df_27[candidates].corrwith(
    df_27['distress']
).abs().sort_values(ascending=False)

print("Top 10 candidate features to potentially add:")
print(candidate_corrs.head(10).round(3).to_string())

Top 10 candidate features to potentially add:
india_gdp_growth_roll3_std         0.511
imports_pct_gdp_lag1               0.505
unemployment_lag1                  0.472
unemployment_lag2                  0.471
gdp_growth_chg3                    0.465
imports_pct_gdp_roll3_mean         0.451
gdp_growth_roll3_mean              0.410
forex_reserves_months_roll3_std    0.396
gross_investment_gdp_lag2          0.396
consumption_growth_lag1            0.394


In [ ]:
# ================================================
# TEST ADDING 1-2 FEATURES TO CURRENT 5
# ================================================

# Align all datasets to same index
common_idx = df_current.dropna().index.intersection(
    df_27.dropna().index
)

df_base5 = df_current.loc[common_idx].copy()
df_extra  = df_27.loc[common_idx].copy()

X_base = df_base5.drop(columns=['distress'])
y      = df_base5['distress']

xgb = XGBClassifier(
    n_estimators=100, max_depth=3,
    learning_rate=0.1, scale_pos_weight=3,
    random_state=42, eval_metric='logloss',
    verbosity=0
)

# Top 5 candidates to test adding
top_candidates = [
    'imports_pct_gdp_lag1',
    'gdp_growth_chg3',
    'imports_pct_gdp_roll3_mean',
    'gdp_growth_roll3_mean',
    'forex_reserves_months_roll3_std',
]

results = []

# Baseline — current 5 features
r0 = run_loocv(X_base, y, xgb, "Baseline (5 features)")
results.append({**r0, 'features': 5})

# Test adding each candidate one at a time
print("\n--- Adding 1 feature at a time ---")
for cand in top_candidates:
    X_test = X_base.copy()
    X_test[cand] = df_extra[cand]
    X_test = X_test.dropna()
    y_test = y.loc[X_test.index]
    r = run_loocv(X_test, y_test, xgb, f"5 + {cand}")
    results.append({**r, 'features': 6})

# Test best 2 combinations
print("\n--- Adding 2 features at a time ---")
best_single = 'imports_pct_gdp_lag1'
for cand in top_candidates:
    if cand == best_single:
        continue
    X_test = X_base.copy()
    X_test[best_single] = df_extra[best_single]
    X_test[cand]        = df_extra[cand]
    X_test = X_test.dropna()
    y_test = y.loc[X_test.index]
    r = run_loocv(X_test, y_test, xgb,
                  f"5 + {best_single} + {cand}")
    results.append({**r, 'features': 7})

# Summary table
print("\n" + "="*65)
print("  FULL COMPARISON")
print("="*65)
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('f1', ascending=False)
for _, row in results_df.iterrows():
    marker = "🏆" if row['f1'] == results_df['f1'].max() else "  "
    print(f"{marker} F1={row['f1']:.3f}  "
          f"BalAcc={row['bal']:.3f}  "
          f"Recall={row['rec']:.3f}  "
          f"FA={row['fa']}  "
          f"({row['features']}feat)  "
          f"{row['label']}")


Baseline (5 features)
  F1=0.714  BalAcc=0.814  Recall=0.714  FalseAlarms=2

--- Adding 1 feature at a time ---

5 + imports_pct_gdp_lag1
  F1=0.615  BalAcc=0.742  Recall=0.571  FalseAlarms=2

5 + gdp_growth_chg3
  F1=0.533  BalAcc=0.699  Recall=0.571  FalseAlarms=4

5 + imports_pct_gdp_roll3_mean
  F1=0.571  BalAcc=0.720  Recall=0.571  FalseAlarms=3
